# Lung cancer pipeline

Extracts clinical entities from the notes of lung cancer patients with the model solarte7/deepLungCancer, BERT version, validates the TNM and saves a CSV and one chart per patient.

**Requirements:** Python 3.12 and the packages of requirements_pulmon_and_prostata.txt.

**What you have to do:** fill in the paths of the configuration cell and run all the cells in order.

## 1. Configuration

**Fill in these lines:**

- ENTRADA: the path to process: the folder of the lung cancer patients (one subfolder per patient), one patient's folder, or one .txt note.
- CARPETA_SALIDA_BASE: a local folder where the CSV and the charts are saved.
- CARPETA_MODELOS: the folder with the models (by default, modelos next to this notebook).

In [ ]:
# offline: set before any import that could reach Hugging Face
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import datetime
import re
import warnings
from pathlib import Path

import pandas as pd

TIPO_CANCER = "pulmon"            # fixed in this notebook
ENTRADA = r""                    # <-- paste: the folder of the lung patients, one patient's folder, or one note (.txt)
CARPETA_SALIDA_BASE = r""        # <-- local folder outside OneDrive; outputs go to outputs/AAAA-MM-DD_pipeline_pulmon/
CARPETA_MODELOS = Path("modelos")    # <-- folder with the models (default: next to this notebook)


def inside_onedrive(path):
    """True if the path falls inside the folder that OneDrive syncs."""
    path = Path(path).resolve()
    roots = [os.environ.get(v) for v in ("OneDrive", "OneDriveConsumer", "OneDriveCommercial")]
    return (any(r and path.is_relative_to(Path(r).resolve()) for r in roots)
            or any(part.lower().startswith("onedrive") for part in path.parts))


def revisar_configuracion():
    """Check both paths before reading any note and create the output folder. Prints no folder or file names."""
    if TIPO_CANCER != "pulmon":
        raise ValueError("This is the lung notebook: TIPO_CANCER must be 'pulmon'")
    if not ENTRADA or not Path(ENTRADA).exists():
        raise FileNotFoundError("Set ENTRADA to an existing path: the lung folder, one patient's folder or one note")
    if not CARPETA_SALIDA_BASE:
        raise ValueError("Set CARPETA_SALIDA_BASE to a local folder outside OneDrive")
    salida = Path(CARPETA_SALIDA_BASE) / "outputs" / f"{datetime.date.today().isoformat()}_pipeline_{TIPO_CANCER}"
    if inside_onedrive(salida):
        raise ValueError("CARPETA_SALIDA_BASE is inside OneDrive: use a local folder")
    salida.mkdir(parents=True, exist_ok=True)
    print(f"cancer type: {TIPO_CANCER} | input: {'a file' if Path(ENTRADA).is_file() else 'a folder'}")
    print("output folder ready, outside OneDrive")
    return salida

In [ ]:
SALIDA = revisar_configuracion()

## 2. Data loading

Lists the notes in ENTRADA. The patient is the name of its folder and the note is the file name without .txt. Prints how many patients and notes were found.

In [ ]:
NOMBRE_NOTA = re.compile(r"(\d+)\s*(.*?)\s*(?:_\d+)?")   # type code (625, 807...), type text, _N suffix


def orden_natural(nombre):
    """Sort key so that 'Nota_2.txt' goes before 'Nota_10.txt'."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", nombre)]


def leer_texto(ruta):
    """Hospital .txt files are not always UTF-8."""
    for codificacion in ("utf-8-sig", "cp1252"):
        try:
            return Path(ruta).read_text(encoding=codificacion)
        except UnicodeDecodeError:
            continue
    return Path(ruta).read_text(encoding="latin-1")


def leer_nombre_archivo(nombre_archivo):
    """'625Apoyo Dx Patología_25.txt' -> ('625Apoyo Dx Patología_25', 'Apoyo Dx Patología'): nota_id (the name
    without .txt) and the type (the text between the type code and the _N suffix); type '' if the name does not fit."""
    nota_id = Path(nombre_archivo).stem
    m = NOMBRE_NOTA.fullmatch(nota_id)
    return nota_id, (m[2] or "tipo desconocido") if m else ""


def seleccionar_notas(entrada):
    """Table of the notes to process and the mode, from what was pasted in ENTRADA: a .txt note ("una_nota"),
    a patient folder with .txt notes ("un_paciente") or the folder of the cancer type, one subfolder per patient
    ("todos"). A note is its patient + its nota_id. Prints and error messages show no folder or file names."""
    entrada = Path(entrada)
    if entrada.is_file():
        if entrada.suffix.lower() != ".txt":
            raise ValueError("ENTRADA is a file, but not a .txt note")
        modo, raiz, pacientes = "una_nota", entrada.parent.parent, [entrada.parent.name]
    else:
        tiene_notas = any(entrada.glob("*.txt"))
        subcarpetas = sorted((p.name for p in entrada.iterdir() if p.is_dir()), key=orden_natural)
        if tiene_notas and subcarpetas:
            raise ValueError("ENTRADA has .txt notes and also subfolders: use one patient's folder "
                             "or the folder of the cancer type")
        if tiene_notas:
            modo, raiz, pacientes = "un_paciente", entrada.parent, [entrada.name]
        elif subcarpetas:
            modo, raiz, pacientes = "todos", entrada, subcarpetas
        else:
            raise FileNotFoundError("ENTRADA has no .txt notes and no patient subfolders")

    filas = []
    for p in pacientes:
        for orden, archivo in enumerate(sorted((raiz / p).glob("*.txt"), key=lambda a: orden_natural(a.name)), 1):
            nota_id, tipo = leer_nombre_archivo(archivo.name)
            filas.append({"paciente_id": p, "nota_id": nota_id, "tipo_nota": tipo, "orden_nota": orden, "ruta": archivo})
    notas = pd.DataFrame(filas, columns=["paciente_id", "nota_id", "tipo_nota", "orden_nota", "ruta"])
    if modo == "una_nota":
        notas = notas[[entrada.samefile(r) for r in notas["ruta"]]].reset_index(drop=True)
    if notas.empty:
        raise FileNotFoundError("No .txt note found in ENTRADA")

    print(f"mode: {modo} | patients: {notas['paciente_id'].nunique()} | notes: {len(notas)}")
    print("notes per type:", notas["tipo_nota"].replace("", "no type (name does not fit)").value_counts().to_dict())
    print(f"file names that do not fit: {int((notas['tipo_nota'] == '').sum())} | notes whose nota_id is "
          f"also in another patient (counted apart): {int(notas.duplicated('nota_id', keep=False).sum())}")
    return notas, modo

In [ ]:
# only counts are printed
notas, MODO = seleccionar_notas(ENTRADA)

## 3. Model loading

Loads the model from its folder and checks it is the right one: complete weights, sha256 fingerprint and labels.

In [ ]:
import hashlib
import importlib.metadata
import platform

import torch
from IPython.display import display
from transformers import AutoModelForTokenClassification, AutoTokenizer

MODELO = "solarte7/deepLungCancer (BERT)"
NOMBRE_MODELO = "solarte7/deepLungCancer, BERT (Solarte-Pabón et al., 2022)"
RUTA_MODELO = CARPETA_MODELOS / "deepLungCancer-BERT"   # converted model (see conversion_pulmon/)
# the 19 entities of the article -> the label the model uses
ENTIDADES_ARTICULO = {
    "Cancer concept": "CANCER_CONCEPT", "Stage grouping": "STAGE", "TNM": "TNM", "Family": "FAMILY",
    "Temporal expression – Explicit": "EXPLICIT_DATE", "Temporal expression – Implicit": "IMPLICIT_DATE",
    "Patient events": "OCURRENCE_EVENT", "Toxic habits": "TOXIC_HABIT", "Quantity": "QUANTITY", "Metric": "METRIC",
    "Cancer biomarkers": "BIOMARKER", "Biomarker status": "BIOMARKER_STATUS", "Patient comorbidities": "COMORBIDITY",
    "Clinical service": "CLINICAL_SERVICE", "Drug": "DRUG", "Surgery": "SURGERY", "Frequency": "FREQ",
    "Chemotherapy": "CHEMOTHERAPY", "Radiotherapy": "RADIOTHERAPY",
}
# reference fingerprint of the converted weights (hf/model.safetensors): there is no published one, because this file
# was made by converting the author's TensorFlow weights (2026-10-19; per-tensor CRC32C of the author's index: 201 of
# 201 equal). It was validated against the original model run in TensorFlow 2.4.1 on 2026-11-03: same label in 616 of
# 616 pieces of 20 invented sentences, largest probability difference 5.3e-06
SHA256_REFERENCIA = "fadf9bf51399823ec12d8f9111d63f0b38d4a31571e1b27bf3e012861ee51d76"


def cargar_modelo(ruta):
    """Tokenizer and model from the folder (no internet). Also returns which weights were loaded."""
    tokenizador = AutoTokenizer.from_pretrained(ruta)
    modelo, carga = AutoModelForTokenClassification.from_pretrained(ruta, output_loading_info=True)
    modelo.eval()                                       # inference: no dropout
    return tokenizador, modelo, carga


def huella(ruta_archivo):
    h = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def verificar_modelo(ruta, tokenizador, modelo, carga):
    """Print what was loaded: path, architecture, weights, fingerprint, labels."""
    print(f"Path.home(): {Path.home()}")
    print(f"model path: {ruta} | exists: {ruta.is_dir()}")
    config = modelo.config
    print(f"arquitectura: {type(modelo).__name__} (model_type = {config.model_type}) -> BERT, not BiLSTM-CRF | "
          f"{config.num_hidden_layers} layers, vocabulary of {config.vocab_size}, {config.num_labels} outputs | "
          f"fast tokenizer: {tokenizador.is_fast}")
    faltaron, sobraron, otra_forma = carga["missing_keys"], carga["unexpected_keys"], carga["mismatched_keys"]
    print(f"weights: {len(modelo.state_dict()) - len(faltaron)} of {len(modelo.state_dict())} loaded | missing: "
          f"{len(faltaron)} | unexpected: {len(sobraron)} | other shape: {len(otra_forma)}")
    if faltaron or sobraron or otra_forma:              # a layer without its weights would be left random
        raise ValueError("The model did not load completely: a layer would keep random weights")
    sha = huella(ruta / "model.safetensors")
    print(f"sha256 of model.safetensors: {sha} | same as the conversion validated against TensorFlow: "
          f"{'Yes' if sha == SHA256_REFERENCIA else 'No'}")
    if sha != SHA256_REFERENCIA:                        # not the validated conversion: stop before using it
        raise ValueError("The weights file is not the conversion validated against TensorFlow")

    etiquetas = list(config.id2label.values())
    con_b = {e[2:] for e in etiquetas if e.startswith("B-")}
    con_i = {e[2:] for e in etiquetas if e.startswith("I-")}
    print(f"outputs: {len(con_b)} B- + {len(con_i)} I- + O + {etiquetas.count('O') - 1} special ones counted as O")
    tabla = pd.DataFrame([{"entity of the article": a, "model label": e, "in the model": "Yes" if e in con_b else "No",
                           "has I-": "Yes" if e in con_i else "No"} for a, e in ENTIDADES_ARTICULO.items()])
    faltantes = [a for a, e in ENTIDADES_ARTICULO.items() if e not in con_b]
    print(f"entities of the article missing in the model: {faltantes or 'none'} | "
          f"model labels not in the article: {sorted(con_b - set(ENTIDADES_ARTICULO.values())) or 'none'}")
    display(tabla)

    return sha


def registrar_versiones(carpeta, ruta, sha):
    """One row in registro_versiones.csv (name, path, fingerprint and library versions)."""
    fila = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "tipo_cancer": TIPO_CANCER,
            "modelo": MODELO, "nombre": NOMBRE_MODELO, "ruta": str(ruta), "sha256": sha, "python": platform.python_version(),
            **{p: importlib.metadata.version(p) for p in ("transformers", "torch", "tokenizers", "pandas", "matplotlib")}}
    archivo = carpeta / "registro_versiones.csv"
    previo = pd.read_csv(archivo, sep=";", encoding="utf-8-sig", dtype=str) if archivo.exists() else pd.DataFrame()
    pd.concat([previo, pd.DataFrame([fila])]).to_csv(archivo, index=False, sep=";", encoding="utf-8-sig")
    print(f"registro_versiones.csv: {len(previo) + 1} rows")

In [ ]:
tokenizador, modelo, carga = cargar_modelo(RUTA_MODELO)
SHA_MODELO = verificar_modelo(RUTA_MODELO, tokenizador, modelo, carga)
registrar_versiones(SALIDA, RUTA_MODELO, SHA_MODELO)

## 4. Sentence segmentation

PyRuSH (the rule-based sentence splitter of medSpaCy, with its Spanish rules in rush_rules_es.tsv) splits each note into sentences. Each sentence keeps where it starts in the note, so the entities can be placed in the note.

Prints how many notes and sentences were found.

In [ ]:
from collections import Counter

from loguru import logger
from PyRuSH import RuSH

REGLAS_ORACIONES = Path("rush_rules_es.tsv")    # PyRuSH rules for Spanish, copied from medSpaCy 1.3.1 (resources/es)
if not REGLAS_ORACIONES.exists():
    raise FileNotFoundError("rush_rules_es.tsv is missing in the notebook folder (Spanish PyRuSH rules)")
logger.disable("PyRuSH")                          # its debug messages can print the text of the notes
SEGMENTADOR = RuSH(rules=str(REGLAS_ORACIONES), enable_logger=False)


def dividir_oraciones(texto):
    """[(start, sentence)]: the sentences PyRuSH finds (rule-based sentence splitter of medSpaCy, Spanish rules)."""
    return [(s.begin, texto[s.begin:s.end]) for s in SEGMENTADOR.segToSentenceSpans(texto)
            if texto[s.begin:s.end].strip()]


def dividir_notas(notas):
    """Table of the sentences of every note: patient, note, start of the sentence in the note and its text.
    Prints counts only."""
    filas, errores = [], Counter()
    for nota in notas.itertuples():
        try:
            for desde, oracion in dividir_oraciones(leer_texto(nota.ruta)):
                filas.append({"paciente_id": nota.paciente_id, "nota_id": nota.nota_id, "inicio": desde,
                              "oracion": oracion})
        except Exception as error:               # counted, without the note name, and the run goes on
            errores[type(error).__name__] += 1
    oraciones = pd.DataFrame(filas, columns=["paciente_id", "nota_id", "inicio", "oracion"])
    print(f"sentence segmentation: {len(notas)} notes | {len(oraciones)} sentences | errors: {dict(errores) or 0}")
    return oraciones

In [ ]:
# real notes: only counts are printed
oraciones = dividir_notas(notas)

## 5. Named entity recognition

For each note, on its sentences, as in the author's code (loadBERT.ipynb):

1. **Tokenization:** words are split by spaces and each one into WordPiece pieces.
2. **Token classification:** the model labels each piece.
3. **Subword merging:** each word takes the label of its first piece.
4. **Entity grouping:** consecutive B-/I- words are joined into one entity, with its position in the note.

Prints how many notes and entities were found.

In [ ]:
LIMITE_PIEZAS = 510                   # 512 minus [CLS] and [SEP]
PUNTUACION_FINAL = " \n\t.,;:"


def tokenizar(tokenizador, oracion):
    """Words split by spaces (as in the author's code) and their WordPiece pieces, in chunks of up to 510 pieces.
    Returns the words [(start, end)] and the chunks [(encoding, word indices)]."""
    palabras = [(m.start(), m.end()) for m in re.finditer(r"\S+", oracion)]
    textos = [oracion[i:f] for i, f in palabras]
    trozos, actual, piezas = [], [], 0
    for k, t in enumerate(textos):
        n = max(1, len(tokenizador.tokenize(t)))
        if actual and piezas + n > LIMITE_PIEZAS:
            trozos.append(actual)
            actual, piezas = [], 0
        actual.append(k)
        piezas += n
    if actual:
        trozos.append(actual)
    return palabras, [(tokenizador([textos[k] for k in t], is_split_into_words=True, truncation=True,
                                   max_length=LIMITE_PIEZAS + 2, return_tensors="pt"), t) for t in trozos]


def predecir(modelo, codificado):
    """Probability of the 40 outputs for each piece."""
    with torch.no_grad():
        return torch.softmax(modelo(**codificado).logits[0], dim=-1)


def unir_subpalabras(modelo, probabilidades, codificado, indices):
    """{word index: label}: each word takes the label of its first piece."""
    etiquetas, vistas = {}, set()
    for posicion, w in enumerate(codificado.word_ids(0)):
        if w is not None and w not in vistas:
            vistas.add(w)
            etiquetas[indices[w]] = modelo.config.id2label[int(probabilidades[posicion].argmax())]
    return etiquetas


def agrupar_entidades(etiquetas, palabras, oracion, desde):
    """B-/I- words in a row -> entities [(label, start, end)] on the note, without trailing punctuation."""
    grupos, actual = [], None
    for k, (i, f) in enumerate(palabras):
        etiqueta = etiquetas.get(k, "O")
        if etiqueta == "O":
            actual = None
        elif etiqueta.startswith("I-") and actual is not None and actual[0] == etiqueta[2:]:
            actual[2] = f                              # the entity goes on
        else:                                          # B-, or I- without its B-: a new entity
            actual = [etiqueta[2:], i, f]
            grupos.append(actual)
    entidades = []
    for etiqueta, i, f in grupos:
        while f - 1 > i and oracion[f - 1] in PUNTUACION_FINAL:
            f -= 1
        entidades.append((etiqueta, desde + i, desde + f))
    return entidades


def reconocer_entidades(oraciones, tokenizador, modelo):
    """All the steps on the sentences of a note [(start, sentence)]: [(label, start, end)] on the note."""
    entidades = []
    for desde, oracion in oraciones:
        palabras, trozos = tokenizar(tokenizador, oracion)
        for codificado, indices in trozos:
            etiquetas = unir_subpalabras(modelo, predecir(modelo, codificado), codificado, indices)
            entidades += agrupar_entidades(etiquetas, [palabras[k] for k in indices], oracion, desde)
    return entidades

In [ ]:
import time
from collections import Counter

CARACTERES_CONTEXTO = 100


def limpio(texto):
    return " ".join(texto.split())             # no line breaks inside the CSV


def correr_ner(notas, oraciones):
    """Entities (all the labels of the model) of every note, with their position in the note. Prints counts only."""
    filas, errores, inicio_reloj = [], Counter(), time.perf_counter()
    por_nota = {clave: list(zip(g["inicio"], g["oracion"]))
                for clave, g in oraciones.groupby(["paciente_id", "nota_id"], sort=False)}
    for nota in notas.itertuples():
        try:
            texto = leer_texto(nota.ruta)
            entidades = reconocer_entidades(por_nota.get((nota.paciente_id, nota.nota_id), []), tokenizador, modelo)
            for etiqueta, i, f in entidades:
                filas.append({"paciente_id": nota.paciente_id, "nota_id": nota.nota_id, "etiqueta": etiqueta,
                              "texto_extraido": limpio(texto[i:f]), "inicio": i, "fin": f,
                              "contexto": limpio(texto[max(0, i - CARACTERES_CONTEXTO):f + CARACTERES_CONTEXTO])})
        except Exception as error:               # counted, without the note name, and the run goes on
            errores[type(error).__name__] += 1
    extracciones = pd.DataFrame(filas, columns=["paciente_id", "nota_id", "etiqueta", "texto_extraido", "inicio", "fin",
                                                "contexto"])
    print(f"recognition: {len(notas)} notes in {time.perf_counter() - inicio_reloj:.1f} s | {len(extracciones)} "
          f"entities | errors: {dict(errores) or 0}")
    return extracciones


# real notes: only counts are printed
EXTRACCIONES_TODAS = correr_ner(notas, oraciones)
etiquetas_modelo = sorted({e[2:] for e in modelo.config.id2label.values() if e != "O"})

## 6. Note date

The date of each note is the first EXPLICIT_DATE entity with a real day, month and year. It is taken from the entities of all the labels, before the label filter, because the date labels are not in the specialists scheme. A note without one stays as "sin fecha" and goes last in the chart.

Prints how many notes have a date.

In [ ]:
from collections import Counter

# date of the note (the criterion of the chart notebook)
ETIQUETA_FECHA = "EXPLICIT_DATE"
SIN_FECHA = "sin fecha"
MESES = {"enero": 1, "ene": 1, "febrero": 2, "feb": 2, "marzo": 3, "mar": 3, "abril": 4, "abr": 4,
         "mayo": 5, "may": 5, "junio": 6, "jun": 6, "julio": 7, "jul": 7, "agosto": 8, "ago": 8,
         "septiembre": 9, "setiembre": 9, "sept": 9, "sep": 9, "set": 9, "octubre": 10, "oct": 10,
         "noviembre": 11, "nov": 11, "diciembre": 12, "dic": 12}
NOMBRES_MES = "|".join(sorted(MESES, key=len, reverse=True))
FECHA_AMD = re.compile(r"(?<!\d)(\d{4})\s*[/.-]\s*(\d{1,2})\s*[/.-]\s*(\d{1,2})(?!\d)")              # 2024-03-12
FECHA_DMA = re.compile(r"(?<!\d)(\d{1,2})\s*[/.-]\s*(\d{1,2})\s*[/.-]\s*(\d{4}|\d{2})(?!\d)")        # 12/03/2024
FECHA_CON_MES = re.compile(rf"(?<!\d)(\d{{1,2}})\s*(?:de\s+|[/.-]\s*)?({NOMBRES_MES})\.?\s*(?:del?\s+|[/.-]\s*)?"
                           rf"(\d{{4}})(?!\d)", re.IGNORECASE)                                       # 12 de marzo de 2024


def fecha_completa(texto):
    """'AAAA-MM-DD' if the text has a real date with day, month and year; '' otherwise."""
    if m := FECHA_AMD.search(texto):
        anio, mes, dia = m.groups()
    elif m := FECHA_DMA.search(texto) or FECHA_CON_MES.search(texto):
        dia, mes, anio = m.groups()
    else:
        return ""
    mes = int(mes) if mes.isdigit() else MESES[mes.lower()]
    anio = int(anio)
    if anio < 100:                             # two-digit year: 24 -> 2024, 58 -> 1958
        anio += 2000 if anio <= datetime.date.today().year % 100 else 1900
    try:
        return datetime.date(anio, mes, int(dia)).isoformat()
    except ValueError:                         # 31/02/2024, month 13...
        return ""


def fecha_de_la_nota(entidades, texto):
    """The first EXPLICIT_DATE (by start) that converts to day/month/year; if none, the note goes by order."""
    fechas = sorted((i, f) for etiqueta, i, f in entidades if etiqueta == ETIQUETA_FECHA)
    for k, (i, f) in enumerate(fechas):
        if fecha := fecha_completa(texto[i:f]):
            return {"fecha_nota": fecha, "origen_fecha": ETIQUETA_FECHA, "texto_fecha": texto[i:f],
                    "inicio_fecha": i, "no_uso_primera": k > 0}
    return {"fecha_nota": SIN_FECHA, "origen_fecha": "orden", "texto_fecha": "", "inicio_fecha": "",
            "no_uso_primera": bool(fechas)}


def fechar_notas(notas, extracciones):
    """One row per note with its date, from the entities of all the labels. Prints counts only."""
    por_nota = {clave: [(e, int(i), int(f)) for e, i, f in zip(g["etiqueta"], g["inicio"], g["fin"])]
                for clave, g in extracciones.groupby(["paciente_id", "nota_id"], sort=False)}
    filas, errores = [], Counter()
    for nota in notas.itertuples():
        try:
            filas.append({"paciente_id": nota.paciente_id, "nota_id": nota.nota_id,
                          **fecha_de_la_nota(por_nota.get((nota.paciente_id, nota.nota_id), []), leer_texto(nota.ruta))})
        except Exception as error:               # counted, without the note name, and the run goes on
            errores[type(error).__name__] += 1
    fechas = pd.DataFrame(filas, columns=["paciente_id", "nota_id", "fecha_nota", "origen_fecha", "texto_fecha",
                                          "inicio_fecha", "no_uso_primera"])
    print(f"note dates: {len(notas)} notes | with date: {int((fechas['origen_fecha'] == ETIQUETA_FECHA).sum())} | "
          f"without date: {int((fechas['origen_fecha'] == 'orden').sum())} | errors: {dict(errores) or 0}")
    return fechas

In [ ]:
# real notes: only counts are printed
fechas = fechar_notas(notas, EXTRACCIONES_TODAS)

## 7. Label filtering

Keeps only the selected labels (CANCER_CONCEPT, Tumor Stage (TNM and STAGE), Cancer Biomarkers (BIOMARKER and BIOMARKER_STATUS), SURGERY). The date of each note is kept.

In [ ]:
# labels of the specialists' scheme: the only entities that go to the CSV and the chart
ETIQUETAS_SELECCIONADAS = ["CANCER_CONCEPT", "STAGE", "TNM", "BIOMARKER", "BIOMARKER_STATUS", "SURGERY"]


def filtrar_etiquetas(extracciones, etiquetas):
    """Only the rows of the selected labels. The date of each note was taken before, from all the entities, so it
    stays in fechas even if its label is not selected."""
    return extracciones[extracciones["etiqueta"].isin(etiquetas)].reset_index(drop=True)


fuera_del_modelo = sorted(set(ETIQUETAS_SELECCIONADAS) - set(etiquetas_modelo))
if fuera_del_modelo:
    raise ValueError(f"These selected labels are not in the model: {fuera_del_modelo}")
extracciones = filtrar_etiquetas(EXTRACCIONES_TODAS, ETIQUETAS_SELECCIONADAS)
print(f"selected labels: {len(ETIQUETAS_SELECCIONADAS)} | entities: {len(extracciones)} of "
      f"{len(EXTRACCIONES_TODAS)} | per label: "
      f"{extracciones['etiqueta'].value_counts().reindex(ETIQUETAS_SELECCIONADAS, fill_value=0).to_dict()}")

## 8. TNM validation

Checks each TNM against the AJCC categories of this cancer: it is valid if it has T, N and M together. It also splits it into T, N and M.

In [ ]:
# split of a TNM text into prefix, T, N and M (the same for every cancer; validity is decided below with the
# AJCC categories of this cancer)
PREFIJO = r"(?:yc|yp|rc|rp|mr|c|p|r|y|a|u)"
EJE_T = r"T(?:is|x|0|[1-4](?:mi|[a-d])?)"
EJE_N = r"N(?:x|0|[1-3](?:mi|[a-c])?)(?:\((?:sn|i\+)\))?"
EJE_M = r"M(?:x|0|1(?:a|b|c[12]?)?)"
SUELTAS = r"[a-z]{0,3}?"          # letters that fit no category: if present, the TNM is irregular

PATRON_TNM = (
    r"(?<![a-z0-9])"
    rf"(?P<prefijo>{PREFIJO})?(?P<T>{EJE_T})"
    rf"(?P<entre_TN>{SUELTAS})\s*(?P<prefijo_N>{PREFIJO}(?=N))?(?P<N>{EJE_N})"
    rf"(?:(?P<entre_NM>{SUELTAS})\s*(?P<prefijo_M>{PREFIJO}(?=M))?(?P<M>{EJE_M}))?"
    r"(?P<al_final>[a-z]{0,2})"
    r"(?![a-z0-9])"
)
_PATRON = re.compile(PATRON_TNM, re.IGNORECASE)

# one loose axis, with its prefix if it has one (for what the models label: "N0")
_EJE_SUELTO = {eje: re.compile(rf"(?<![a-z0-9])(?P<prefijo>{PREFIJO})?(?P<valor>{patron})(?![a-z0-9])",
                               re.IGNORECASE)
               for eje, patron in (("T", EJE_T), ("N", EJE_N), ("M", EJE_M))}


def _normalizar(eje):
    """t2C -> T2c, tis -> Tis, mx -> MX, n1(SN) -> N1(sn)."""
    return eje[0].upper() + eje[1:].lower().replace("x", "X")


def _c_pegada(m, eje, siguiente, entre, prefijo_siguiente):
    """The axis ends in c and the next axis comes right after it, without its own prefix."""
    return (m[siguiente] is not None and m[eje][-1:].lower() == "c"
            and not m[entre] and not m[prefijo_siguiente]
            and m.end(eje) == m.start(siguiente))


def _desde_coincidencia(m):
    prefijo = (m["prefijo"] or "").lower()
    T, N = _normalizar(m["T"]), _normalizar(m["N"])
    ambiguo_T = _c_pegada(m, "T", "N", "entre_TN", "prefijo_N")
    ambiguo_N = _c_pegada(m, "N", "M", "entre_NM", "prefijo_M")

    # most likely reading of the ambiguous c: if the rest of the TNM repeats prefixes (some axis has its own prefix,
    # or there is more than one stuck c: cT2cN0cM0), the c is the prefix of the next axis. Otherwise (cT1cN0M0,
    # T4bN3CM1) it is a subcategory, which is more frequent: T1c is very common in breast and lung.
    repite_prefijos = bool(m["prefijo_N"] or m["prefijo_M"]) or (ambiguo_T and ambiguo_N)
    if repite_prefijos:
        T = T[:-1] if ambiguo_T else T
        N = N[:-1] if ambiguo_N else N

    M = _normalizar(m["M"]) if m["M"] else ""
    return {
        "texto": m.group(0),
        "prefijo": prefijo,
        "T": T,
        "N": N,
        "M": M,
        "irregular": bool(m["entre_TN"] or m["entre_NM"] or m["al_final"]),
        "ambiguo": ambiguo_T or ambiguo_N,
        "parcial": not M,
    }


def _ejes_sueltos(fragmento):
    """T, N and M that appear loose in the fragment; None if none can be read."""
    encontrados = {eje: patron.search(fragmento) for eje, patron in _EJE_SUELTO.items()}
    presentes = [m for m in encontrados.values() if m]
    if not presentes:
        return None
    primero = min(presentes, key=lambda m: m.start())
    return {
        "texto": fragmento,
        "prefijo": (primero["prefijo"] or "").lower(),
        **{eje: _normalizar(m["valor"]) if m else "" for eje, m in encontrados.items()},
        "irregular": False,
        "ambiguo": False,
        "parcial": len(presentes) < 3,
    }


def descomponer_tnm(fragmento):
    """Prefix, T, N, M, irregular, ambiguo and parcial of the first TNM of the fragment. If there is no complete TNM,
    it tries loose axes; None if no axis can be read."""
    fragmento = fragmento or ""
    m = _PATRON.search(fragmento)
    return _desde_coincidencia(m) if m else _ejes_sueltos(fragmento)


# AJCC 8th edition, lung
PREFIJO = r"(?:yc|yp|rc|rp|c|p|r|a)?"
REGLAS_TNM = {
    "pulmon": {
        "T": PREFIJO + r"T(?:X|0|is|1mi|1[abc]?|2[ab]?|3|4)",     # TX, T0, Tis, T1mi, T1a-c, T2a-b, T3, T4
        "N": PREFIJO + r"N(?:X|0|1|2|3)",                         # NX, N0-N3
        "M": r"(?:c?M0|[cp]?M1[abc]?)",                           # cM0, cM1a-c, pM1 (MX and pM0 do not exist)
    },
}
SEPARADOR = r"[\s,;/-]*"                         # between two axes only
T, N, M = (REGLAS_TNM[TIPO_CANCER][eje] for eje in "TNM")
TNM_VALIDO = re.compile(rf"{T}{SEPARADOR}{N}{SEPARADOR}{M}", re.IGNORECASE)               # T, N and M together
TNM_INCOMPLETO = re.compile(rf"{T}(?:{SEPARADOR}{N})?(?:{SEPARADOR}{M})?|{N}(?:{SEPARADOR}{M})?|{M}", re.IGNORECASE)


def aplicar_regla_tnm(texto_extraido):
    """valido and motivo from the rule; T, N and M from descomponer_tnm, even if the TNM is not valid."""
    texto = texto_extraido.strip(" .,;:")
    valido = TNM_VALIDO.fullmatch(texto) is not None
    partes = descomponer_tnm(texto) or {}
    return {"T": partes.get("T", ""), "N": partes.get("N", ""), "M": partes.get("M", ""),
            "valido": "Sí" if valido else "No",
            "motivo": "" if valido else "falta_T_N_o_M" if TNM_INCOMPLETO.fullmatch(texto) else "no_es_TNM_AJCC"}

## 9. Output

Saves the CSV (one row per entity) and one row of counts in registro_corridas.csv, in the output folder.

In [ ]:
COLUMNAS_CSV = ["nota_id", "paciente_id", "tipo_nota", "fecha_nota", "etiqueta", "texto_extraido", "inicio", "fin",
                "contexto", "T", "N", "M", "valido", "motivo"]


def armar_tabla(notas, extracciones, fechas):
    """One row per extraction, valid or not: note data + date + the TNM rule on the TNM rows."""
    nota = ["paciente_id", "nota_id"]                      # a note is its patient + its nota_id
    tabla = (extracciones.merge(notas[nota + ["tipo_nota"]], on=nota)
             .merge(fechas.drop(columns="no_uso_primera"), on=nota))
    regla = [aplicar_regla_tnm(t) if e == "TNM" else dict.fromkeys(("T", "N", "M", "valido", "motivo"), "")
             for t, e in zip(tabla["texto_extraido"], tabla["etiqueta"])]
    tabla = pd.concat([tabla.reset_index(drop=True), pd.DataFrame(regla, columns=["T", "N", "M", "valido", "motivo"])],
                      axis=1)
    return tabla[COLUMNAS_CSV]


def guardar_csv(tabla, carpeta):
    archivo = carpeta / f"{TIPO_CANCER}_extracciones.csv"
    tabla.to_csv(archivo, index=False, sep=";", encoding="utf-8-sig")
    tnm = tabla[tabla["etiqueta"] == "TNM"]
    print(f"{archivo.name}: {len(tabla)} rows | TNM: {len(tnm)} (valid {int((tnm['valido'] == 'Sí').sum())}, "
          f"not valid {int((tnm['valido'] == 'No').sum())}) | reasons: {tnm['motivo'].value_counts().to_dict()}")
    return archivo


def registrar_corrida(carpeta, notas, tabla):
    """One row per run in registro_corridas.csv: counts only, no patient or file names."""
    fila = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "tipo_cancer": TIPO_CANCER, "modo": MODO,
            "modelo": NOMBRE_MODELO,
            "pacientes": notas["paciente_id"].nunique(), "notas": len(notas), "entidades": len(tabla)}
    archivo = carpeta / "registro_corridas.csv"
    previo = pd.read_csv(archivo, sep=";", encoding="utf-8-sig", dtype=str) if archivo.exists() else pd.DataFrame()
    pd.concat([previo, pd.DataFrame([fila])]).to_csv(archivo, index=False, sep=";", encoding="utf-8-sig")
    print(f"registro_corridas.csv: {len(previo) + 1} rows | this run: {fila['pacientes']} patients, "
          f"{fila['notas']} notes, {fila['entidades']} entities")

In [ ]:
# real notes: only counts are printed
tabla = armar_tabla(notas, extracciones, fechas)
ARCHIVO_CSV = guardar_csv(tabla, SALIDA)
registrar_corrida(SALIDA, notas, tabla)

## 10. Visualization

### Highlighted entities

One HTML page per patient with all its notes, in date order, and the entities the model extracted highlighted with one color per label.

In [ ]:
import html as html_lib

from spacy import displacy

# labels highlighted in the note text: the selected labels, the same as the CSV and the timeline
ETIQUETAS_RESALTADO = ETIQUETAS_SELECCIONADAS
PALETA_RESALTADO = ["#aedcf5", "#a8e6e2", "#cfd4da", "#c3e6c3", "#f6b8c8", "#fff0a0", "#e2efc0", "#d9c8f0",
                    "#fcd3a8", "#bfd7f5", "#f3c4e8", "#d6e6a8", "#f5d0c5", "#c8ecd9", "#e8dcc0", "#d0d0f5",
                    "#f0e0a8", "#c5e3ee", "#ead1dc", "#d9ead3", "#fde2b8"]


def colores_etiquetas(etiquetas):
    """One soft color per label, always the same for a label of this model."""
    return {e: PALETA_RESALTADO[k % len(PALETA_RESALTADO)] for k, e in enumerate(sorted(etiquetas))}


def resaltar_paciente(paciente_id, entidades, notas, fechas, colores, archivo):
    """displaCy page of one patient: every note (dated notes by date, undated last) with its entities highlighted.
    Saved as HTML, not shown. Titles in Spanish for the HUSI users."""
    nota = ["paciente_id", "nota_id"]
    notas_p = notas[notas["paciente_id"] == paciente_id].merge(fechas[nota + ["fecha_nota"]], on=nota)
    notas_p = (notas_p.assign(sin=notas_p["fecha_nota"] == SIN_FECHA)
               .sort_values(["sin", "fecha_nota", "orden_nota"]))
    de_paciente = entidades[entidades["paciente_id"] == paciente_id]
    documentos = []
    for n in notas_p.itertuples():
        ents = de_paciente[de_paciente["nota_id"] == n.nota_id].sort_values("inicio")
        documentos.append({
            "text": leer_texto(n.ruta),
            "ents": [{"start": int(i), "end": int(f), "label": e}
                     for e, i, f in zip(ents["etiqueta"], ents["inicio"], ents["fin"])],
            "title": f"{n.nota_id} · {n.tipo_nota or 'sin tipo'} · "
                     f"{'sin fecha' if n.fecha_nota == SIN_FECHA else n.fecha_nota} · {len(ents)} entidades"})
    pagina = displacy.render(documentos, style="ent", manual=True, page=True, jupyter=False,
                             options={"colors": colores})
    titulo = html_lib.escape(f"{paciente_id} · entidades extraídas ({MODELO}, pulmón)")
    pagina = (pagina.replace('<html lang="en">', '<html lang="es">', 1)
              .replace("<head>", '<head>\n<meta charset="utf-8">', 1)          # accents read right in any browser
              .replace("<title>displaCy</title>", f"<title>{titulo}</title>", 1)
              .replace("<figure", f"<h2 style=\"font-family: sans-serif\">{titulo}</h2><figure", 1))
    archivo.write_text(pagina, encoding="utf-8")
    return archivo


In [ ]:
# real patients: the pages are saved, not shown; only counts are printed
entidades_resaltado = filtrar_etiquetas(EXTRACCIONES_TODAS, ETIQUETAS_RESALTADO)
colores_resaltado = colores_etiquetas(ETIQUETAS_RESALTADO)
carpeta_resaltado = SALIDA / "entidades_resaltadas"
carpeta_resaltado.mkdir(exist_ok=True)
guardadas, no_guardadas = 0, 0
for paciente in notas["paciente_id"].unique():
    try:
        nombre = re.sub(r"[^\w.-]+", "_", paciente)       # a safe file name
        resaltar_paciente(paciente, entidades_resaltado, notas, fechas, colores_resaltado,
                          carpeta_resaltado / f"entidades_{nombre}.html")
        guardadas += 1
    except OSError:                       # the HTML is open in another program
        no_guardadas += 1
print(f"highlighted pages saved: {guardadas} | labels: {len(ETIQUETAS_RESALTADO)}"
      + (f" | could not be saved (close them and repeat): {no_guardadas}" if no_guardadas else ""))


### Timeline

One PNG per patient. One row per selected label and one line per value. A bar joins consecutive notes with the same value, with a small mark for every note; it does not mean duration. Soft vertical lines mark every note. Only valid TNM are drawn.

In [ ]:
import unicodedata

import matplotlib.dates as mdates
from matplotlib.figure import Figure              # no pyplot: real charts are never shown in the notebook
from matplotlib.lines import Line2D

ETIQUETAS_GRAFICA = ETIQUETAS_SELECCIONADAS      # the chart draws the selected labels

MARCADORES = ["o", "s", "^", "D", "v", "P", "X", "*", "h", "<", ">"]   # shape of the point = type of note
TAMANO_MARCADOR = {"^": 7, "v": 7, "<": 7, ">": 7, "*": 8}            # so every shape looks the same size (else 6)
COLOR_PUNTO = "#2a78d6"            # one color: the row already says which label it is
COLOR_BARRA = "#9cc2ee"            # bar joining consecutive notes with the same value
VERTICAL = "#e9e9e6"               # soft vertical line at the date of every note
TINTA, TINTA_2, REJILLA, BANDA = "#0b0b0b", "#52514e", "#e4e4e0", "#f4f4f2"
LARGO_VALOR = 45                   # longer values are cut with "…"; also cut so they do not reach the next text
ANCHO_LETRA = 7 * 0.6 / 72         # inches per character at 7 pt (approximate)
ALTO_LINEA, ESPACIO_FILA = 0.21, 0.6   # inches per line (one per distinct value); gap between rows, in lines
NOMBRE_CANCER = "pulmón"


def filas_grafica():
    return list(ETIQUETAS_GRAFICA)


def ubicar_notas(notas_p, fechas):
    """x of each note: its date if every note has one (True); else its place in the order by date, undated last (False)."""
    nota = ["paciente_id", "nota_id"]                      # a note is its patient + its nota_id
    p = notas_p.merge(fechas[nota + ["fecha_nota"]], on=nota).sort_values("orden_nota")
    if (p["fecha_nota"] != SIN_FECHA).all():
        p["x"] = pd.to_datetime(p["fecha_nota"])
        return p.sort_values(["x", "orden_nota"]).reset_index(drop=True), True
    # the file numbers do not follow the dates: dated notes by date, then the undated ones (in file order)
    p = p.assign(sin=p["fecha_nota"] == SIN_FECHA).sort_values(["sin", "fecha_nota", "orden_nota"]).reset_index(drop=True)
    p["x"] = range(1, len(p) + 1)
    return p, False


def clave_valor(valor):
    """Values that differ only in case, accents, spaces or final punctuation are the same value."""
    sin_tildes = "".join(c for c in unicodedata.normalize("NFKD", valor) if not unicodedata.combining(c))
    return re.sub(r"\s+", "", sin_tildes).upper().strip(".,;:")


def puntos_paciente(tabla_p):
    """(nota_id, row, value, key): valid TNM split into T, N and M; every extraction of the other chosen labels.
    The same value once per note and row."""
    puntos = []
    for r in tabla_p.itertuples():
        if r.etiqueta == "TNM" and "TNM" in ETIQUETAS_GRAFICA:
            if r.valido == "Sí":
                puntos.append((r.nota_id, "TNM", r.texto_extraido))      # as the model extracted it
        elif r.etiqueta in ETIQUETAS_GRAFICA:
            puntos.append((r.nota_id, r.etiqueta, r.texto_extraido))
    puntos = pd.DataFrame(puntos, columns=["nota_id", "fila", "valor"])
    puntos["clave"] = puntos["valor"].map(clave_valor)
    return puntos.drop_duplicates(["nota_id", "fila", "clave"])


def ubicar_lineas(puntos, posicion):
    """y of each (row, value): one line per distinct value, in order of first appearance along the axis; rows
    from top to bottom with a gap. Also the (top, bottom) of each row."""
    orden = puntos.assign(pos=puntos["nota_id"].map(posicion)).sort_values("pos", kind="stable")
    y_linea, filas, y = {}, {}, 0.0
    for fila in filas_grafica():
        valores = list(dict.fromkeys(orden.loc[orden["fila"] == fila, "clave"])) or [None]
        for j, v in enumerate(valores):
            y_linea[(fila, v)] = y - j
        filas[fila] = (y, y - len(valores) + 1)
        y -= len(valores) + ESPACIO_FILA
    return y_linea, filas


def graficar_paciente(paciente_id, tabla, notas, fechas, archivo):
    """Timeline of one patient inspired by HeaRT (Gantt-like), saved as PNG. Texts in Spanish for the HUSI users."""
    notas_p, por_fecha = ubicar_notas(notas[notas["paciente_id"] == paciente_id], fechas)
    posicion = {n: k for k, n in enumerate(notas_p["nota_id"])}       # place of each note along the axis
    puntos = puntos_paciente(tabla[tabla["paciente_id"] == paciente_id]).merge(
        notas_p[["nota_id", "x", "tipo_nota", "fecha_nota"]], on="nota_id")
    puntos["pos"] = puntos["nota_id"].map(posicion)
    y_linea, filas = ubicar_lineas(puntos, posicion)
    tipos = sorted(notas_p["tipo_nota"].unique())
    marcador = {t: MARCADORES[k % len(MARCADORES)] for k, t in enumerate(tipos)}

    # size in inches: the plot grows with the number of lines, so no value is drawn on top of another
    y_max, y_min = 0.7, min(abajo for _, abajo in filas.values()) - 0.7
    alto_plot = (y_max - y_min) * ALTO_LINEA
    ejes = 0.75 if por_fecha else 1.15                     # room for the dates (the order axis also has nota_id)
    izquierda, derecha, arriba = 1.9, 0.5, 1.05 + ejes
    abajo_margen = ejes + 0.75                             # dates again below, then the legend
    ancho = 11 if por_fecha else max(8, 1.25 * len(notas_p) + 2.5)
    alto = alto_plot + arriba + abajo_margen
    fig = Figure(figsize=(ancho, alto), dpi=200, facecolor="white")
    ax = fig.add_axes([izquierda / ancho, abajo_margen / alto, (ancho - izquierda - derecha) / ancho, alto_plot / alto])
    if por_fecha:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m/%Y"))
        margen = pd.Timedelta(days=max(15, (notas_p["x"].max() - notas_p["x"].min()).days * 0.06))
        ax.set_xlim(notas_p["x"].min() - margen, notas_p["x"].max() + margen * 3)
    else:
        ax.set_xticks(list(notas_p["x"]), [("s/f" if f == SIN_FECHA else pd.Timestamp(f).strftime("%d/%m/%Y"))
                                           + "\n" + i for f, i in zip(notas_p["fecha_nota"], notas_p["nota_id"])])
        ax.set_xlim(0.5, len(notas_p) + 1.3)
    x_izq, x_der = ax.get_xlim()
    pulgadas = (ancho - izquierda - derecha) / (x_der - x_izq)          # inches per unit of the x axis

    def en_numero(x):
        return mdates.date2num(x) if por_fecha else x

    for k, (fila, (tope, fondo)) in enumerate(filas.items()):
        if k % 2 == 0:                                     # alternate bands group the lines of a row
            ax.axhspan(fondo - 0.5, tope + 0.5, color=BANDA, lw=0, zorder=0)
    for x in notas_p["x"]:                                 # a soft vertical line at every note of the patient
        ax.axvline(x, color=VERTICAL, lw=0.6, zorder=0.5)

    for (fila, clave), grupo in puntos.groupby(["fila", "clave"], sort=False):
        y = y_linea[(fila, clave)]
        grupo = grupo.sort_values("pos")
        presentes = set(grupo["pos"])
        con_texto = [p for p in grupo.itertuples() if p.pos - 1 not in presentes]   # where the value appears
        # Gantt bar: one bar per stretch of consecutive notes with the value
        pos = list(grupo["pos"])
        xs = dict(zip(grupo["pos"], grupo["x"]))
        inicio = 0
        for k in range(1, len(pos) + 1):
            if k == len(pos) or pos[k] != pos[k - 1] + 1:
                if k - inicio > 1:
                    ax.plot([xs[pos[inicio]], xs[pos[k - 1]]], [y, y], color=COLOR_BARRA, lw=5,
                            solid_capstyle="round", zorder=2)
                inicio = k
        bordes = {p for k, p in enumerate(pos) if k == 0 or k == len(pos) - 1
                  or pos[k - 1] != p - 1 or pos[k + 1] != p + 1}
        for p in grupo.itertuples():                       # start and end: full size; notes in between: small
            tamano = TAMANO_MARCADOR.get(marcador[p.tipo_nota], 6) * (1 if p.pos in bordes else 0.55)
            hueco = p.fecha_nota == SIN_FECHA
            ax.plot(p.x, y, marcador[p.tipo_nota], ms=tamano, zorder=3,
                    mfc="white" if hueco else COLOR_PUNTO, mec=COLOR_PUNTO if hueco else "white",
                    mew=1.2 if hueco else 0.8)
        for k, p in enumerate(con_texto):                  # write the value once, cut before the next text
            hasta = en_numero(con_texto[k + 1].x) if k + 1 < len(con_texto) else x_der
            espacio = (hasta - en_numero(p.x)) * pulgadas + (derecha if k + 1 == len(con_texto) else 0) - 0.08
            largo = max(4, min(LARGO_VALOR, int(espacio / ANCHO_LETRA)))
            valor = p.valor if len(p.valor) <= largo else p.valor[:largo - 1] + "…"
            ax.annotate(valor, (p.x, y), xytext=(4, 3.5), textcoords="offset points", fontsize=7, va="bottom",
                        ha="left", color=TINTA, annotation_clip=False, zorder=4)

    ax.set_yticks([tope for tope, _ in filas.values()],
                  list(filas), fontsize=8, color=TINTA_2)
    ax.tick_params(axis="y", length=0)
    ax.set_ylim(y_min, y_max)
    # dates above and below, so they can be read from any part of a tall chart
    ax.tick_params(axis="x", labelsize=7, labelcolor=TINTA_2, color=REJILLA, top=True, labeltop=True)
    for marca in ax.xaxis.get_major_ticks():
        marca.label1.set(rotation=30, ha="right", rotation_mode="anchor")      # below
        marca.label2.set(rotation=30, ha="left", rotation_mode="anchor")       # above
    for lado in ("right", "left"):
        ax.spines[lado].set_visible(False)
    for lado in ("top", "bottom"):
        ax.spines[lado].set_color("#c8c8c4")

    x0 = izquierda / ancho
    fig.text(x0, 1 - 0.38 / alto, f"{paciente_id} · línea de tiempo ({MODELO}, {NOMBRE_CANCER})", fontsize=12,
             fontweight="bold", color=TINTA)
    fig.text(x0, 1 - 0.62 / alto, "Eje: fecha de la nota." if por_fecha else
             "Eje: notas en orden de fecha; las notas sin fecha (s/f) van al final.", fontsize=8, color=TINTA_2)
    fig.text(x0, 1 - 0.82 / alto, "Una línea por valor. TNM: solo los válidos.", fontsize=8, color=TINTA_2)
    leyenda = [Line2D([], [], marker=marcador[t], color=TINTA_2, ls="",
                      label=t or "sin tipo (el nombre no tiene el formato)") for t in tipos]
    leyenda.append(Line2D([], [], marker="o", ls="", mfc="white", mec=TINTA_2, mew=1.2, label="nota sin fecha"))
    fig.legend(handles=leyenda, loc="lower left", bbox_to_anchor=(x0, 0.30 / alto), ncol=min(len(leyenda), 6),
               fontsize=7, frameon=False, title="Tipo de nota", title_fontsize=7, labelcolor=TINTA_2)
    fig.text(x0, 0.12 / alto, "La barra une notas consecutivas en las que aparece el valor; no indica duración.",
             fontsize=7.5, color=TINTA_2)
    fig.savefig(archivo, dpi=200, facecolor="white")
    return archivo

In [ ]:
# real patients: the charts are saved, not shown; only counts are printed
guardadas, no_guardadas = 0, 0
for paciente in notas["paciente_id"].unique():
    try:
        nombre = re.sub(r"[^\w.-]+", "_", paciente)       # a safe file name
        graficar_paciente(paciente, tabla, notas, fechas, SALIDA / f"linea_tiempo_{nombre}.png")
        guardadas += 1
    except OSError:                       # the PNG is open in another program
        no_guardadas += 1
print(f"charts saved: {guardadas}" + (f" | could not be saved (close them and repeat): {no_guardadas}"
                                            if no_guardadas else ""))